In [1]:
import surprise
import numpy as np
import pandas as pd
from surprise.model_selection import train_test_split
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [2]:
ratings = pd.read_csv(r"F:\MACHINE LEARNING\Cases_rec_sys\ml-100k\u.data", sep = '\t', names = ['uid','iid','rating','ts'])
ratings.drop('ts', axis = 1, inplace = True)
ratings.head()

,uid,iid,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [3]:
lowest_rating = ratings['rating'].min()
highest_rating=ratings['rating'].max()
lowest_rating,highest_rating

(1, 5)

In [4]:
reader = surprise.Reader(rating_scale = (lowest_rating, highest_rating))
data = surprise.Dataset.load_from_df(ratings, reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [5]:
trainset, testset = train_test_split(data, test_size= 0.25, random_state=26)
similarity_options = {'name': 'cosine', 'user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k = 40
algo.fit(trainset)
predictions = algo.test(testset)
surprise.accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0253


1.025282928143327

In [8]:
param_grid = {'n_epochs': [50], 'n_factors':[100,50],
              'lr_all': np.linspace(0.001,1,5),
              'reg_all': np.linspace(0.01,0.8,5)
             }
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)
gs = GridSearchCV(surprise.SVD, param_grid , measures = ['rmse','mae'], joblib_verbose= 3,cv = kfold, n_jobs = -1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:   33.0s
[Parallel(n_jobs=-1)]: Done 250 out of 250 | elapsed:  1.2min finished


In [9]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.9467899421542457
{'n_epochs': 50, 'n_factors': 50, 'lr_all': 0.001, 'reg_all': 0.01}


In [10]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

In [11]:
iids = ratings['iid'].unique()
print(len(iids))

1682


In [12]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print('List of items rated by user: \n', u_iid)
print('No. of items rated by user {0}: {1}'.format(user,  len(u_iid)))
iids_to_predict = np.setdiff1d(iids, u_iid)
print("Items not rated by the user or those items for which the expexted ratings are to be predicted: ", iids_to_predict)

List of items rated by user: 
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50: 24
Items not rated by the user or those items for which the expexted ratings are to be predicted:  [   1    2    3 ... 1680 1681 1682]
